<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        🎯 04. Boosting y Casos de Estudio Desbalanceados
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/09%20-%20Decision%20Trees/Para%20Dummies/04_Boosting_y_Casos_Estudio_Desbalanceados_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

En el cuaderno anterior, los árboles de un Random Forest se entrenaban **todos al mismo tiempo, sin comunicarse entre sí** (en paralelo). Ahora veremos una estrategia distinta: entrenar árboles **uno después de otro**, donde cada árbol nuevo se concentra especialmente en corregir los errores del árbol anterior. A esto se le llama **Boosting**.

También vamos a enfrentar un problema muy común en la vida real: ¿qué pasa cuando una de las dos clases que queremos predecir es **mucho más rara** que la otra? (por ejemplo, detectar una enfermedad poco común, o una transacción fraudulenta entre miles de transacciones normales).

Al terminar podrás explicar:
1. En qué se diferencia Boosting de Bagging/Random Forest.
2. Cómo funciona `GradientBoostingClassifier`: aprender de los errores, paso a paso.
3. Por qué un dataset desbalanceado (ej. 90% vs. 10%) engaña a las métricas simples como la exactitud.
4. Cómo `class_weight='balanced'` ayuda a que el modelo le preste más atención a la clase minoritaria.

---
## 1. De un comité en paralelo a un equipo de relevos 🏃

En Bagging y Random Forest, los 100 árboles del comité se entrenan **cada uno por su cuenta**, sin saber lo que hicieron los demás, y al final simplemente se cuentan los votos. Es como pedirle a 100 personas su opinión por separado y luego promediar.

**Boosting** funciona más como una carrera de relevos, donde cada corredor sabe exactamente en qué se equivocó el anterior:

1. El primer árbol (muy simple, casi siempre "débil" a propósito) hace sus predicciones — y comete algunos errores.
2. El **segundo árbol** no empieza desde cero: se entrena prestando especial atención a los casos donde el primer árbol se equivocó.
3. El **tercer árbol** hace lo mismo con los errores que quedan después de sumar los dos primeros. Y así sucesivamente.
4. Al final, la predicción es la **suma acumulada** de las correcciones de todos los árboles del relevo.

> 📌 **Para recordar:** Bagging/Random Forest = comité que opina en paralelo. Boosting = relevo secuencial donde cada corredor corrige los errores del anterior.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

np.random.seed(11)

print("Entorno listo para explorar Boosting y datos desbalanceados.")

---
## 2. Un mini-ejemplo de Boosting "a mano" 🧮

Antes de usar la clase de Scikit-Learn, veamos la idea con un ejemplo muy simple: 8 puntos en una línea, donde queremos separar dos grupos. Vamos a entrenar un primer árbol muy débil (`max_depth=1`, apenas puede hacer **una** pregunta) y ver cuáles puntos clasifica mal — esos serán justamente los que el "segundo corredor del relevo" tendría que corregir.

In [ ]:
x_demo = np.array([1, 2, 3, 4, 5, 6, 7, 8]).reshape(-1, 1)
y_demo = np.array([0, 0, 0, 1, 0, 1, 1, 1])  # un patrón con una excepción en x=3 y x=5

arbol_debil = DecisionTreeClassifier(max_depth=1, random_state=42)
arbol_debil.fit(x_demo, y_demo)
pred_demo = arbol_debil.predict(x_demo)

comparacion = pd.DataFrame({'x': x_demo.ravel(), 'y_real': y_demo, 'prediccion': pred_demo})
comparacion['error'] = comparacion['y_real'] != comparacion['prediccion']
comparacion

### 🤔 ¿Qué acaba de pasar?

- `max_depth=1` obliga al árbol a hacer **una sola pregunta** (por ejemplo, "¿x es mayor que 4.5?"), así que es un modelo intencionalmente débil — un "*stump*" (tocón), como se le conoce en Boosting.
- Ese árbol acierta en la mayoría de los puntos, pero se equivoca en las "excepciones" del patrón (los puntos donde la regla simple no encaja).
- En Boosting real, el **siguiente árbol** del relevo se entrenaría dándole más peso justamente a esas filas donde `error` es `True` — así, el segundo árbol se especializa en corregir lo que el primero no pudo. `GradientBoostingClassifier` automatiza exactamente esta idea, sumando muchos árboles débiles como este uno tras otro.

---
## 3. Simulamos un problema desbalanceado: detectar una condición rara 🧪

Ahora simulamos un caso muy común en la práctica: un examen médico donde el 90% de los pacientes están sanos y solo el 10% tiene una condición rara que queremos detectar. Creamos dos características (`indicador_1`, `indicador_2`) que ayudan un poco a distinguir a los pacientes con la condición, pero con mucho solapamiento (como pasa en la vida real).

In [ ]:
n_sanos = 360
n_enfermos = 40  # aproximadamente 90% / 10%, como en muchos problemas reales

indicador_1_sanos = np.random.normal(5, 1.5, n_sanos)
indicador_2_sanos = np.random.normal(5, 1.5, n_sanos)

indicador_1_enfermos = np.random.normal(7, 1.5, n_enfermos)
indicador_2_enfermos = np.random.normal(7, 1.5, n_enfermos)

pacientes = pd.DataFrame({
    'indicador_1': np.concatenate([indicador_1_sanos, indicador_1_enfermos]),
    'indicador_2': np.concatenate([indicador_2_sanos, indicador_2_enfermos]),
    'condicion': np.concatenate([np.zeros(n_sanos), np.ones(n_enfermos)]).astype(int)
})

print(pacientes['condicion'].value_counts())
print(f"Proporción de la clase minoritaria (enfermos): {pacientes['condicion'].mean()*100:.1f}%")

### 🤔 ¿Qué acaba de pasar?

- Creamos 360 pacientes "sanos" y solo 40 "con la condición" — una proporción cercana a 90% / 10%, típica de problemas como detección de fraude o enfermedades raras.
- Los indicadores de los pacientes enfermos tienen un promedio más alto que los de los sanos, pero con bastante superposición (`np.random.normal` con la misma desviación estándar) — de nuevo, así son los datos reales: rara vez hay una frontera perfecta.
- `value_counts()` confirma el desbalance: hay 9 pacientes sanos por cada paciente con la condición.

---
## 4. La trampa de la exactitud (*accuracy*) con clases desbalanceadas ⚠️

Aquí viene el problema clásico. Si un modelo **simplemente dice "sano" siempre**, sin fijarse en ningún dato, ¿qué tan seguido acertaría?

In [ ]:
X = pacientes[['indicador_1', 'indicador_2']]
y = pacientes['condicion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Un modelo "perezoso" que siempre predice la clase mayoritaria (0 = sano)
prediccion_perezosa = np.zeros_like(y_test)

exactitud_perezosa = accuracy_score(y_test, prediccion_perezosa)
recall_perezoso = recall_score(y_test, prediccion_perezosa)

print(f"Exactitud del modelo 'siempre di sano': {exactitud_perezosa*100:.1f}%")
print(f"Recall (capacidad de detectar enfermos): {recall_perezoso*100:.1f}%")

### 🤔 ¿Qué acaba de pasar?

- El modelo "perezoso" ni siquiera mira los datos: siempre responde "sano". Aun así, su **exactitud** sale muy alta (≈ 90%), simplemente porque el 90% de los pacientes en verdad están sanos.
- Pero su **recall** (qué proporción de los enfermos realmente detectó) es **0%** — no detectó ni un solo caso real de la condición, que es justo lo que más nos importa en un examen médico.
- Esta es la trampa clásica de los datasets desbalanceados: la exactitud puede verse muy bien y aun así el modelo ser inútil para el objetivo real. Por eso, cuando una clase es rara, el **recall de la clase minoritaria** (o el F1-Score) suele ser una métrica mucho más honesta que la exactitud.

---
## 5. Gradient Boosting normal: ¿cae en la misma trampa? 🌳➡🌳➡🌳

Entrenemos ahora un `GradientBoostingClassifier` real (nuestro "equipo de relevos") sobre estos pacientes, sin decirle nada especial sobre el desbalance, y midamos su exactitud y su recall.

In [ ]:
gb_normal = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=42)
gb_normal.fit(X_train, y_train)
pred_normal = gb_normal.predict(X_test)

exactitud_normal = accuracy_score(y_test, pred_normal)
recall_normal = recall_score(y_test, pred_normal)

print(f"Gradient Boosting normal  -> Exactitud: {exactitud_normal*100:.1f}%  |  Recall (enfermos detectados): {recall_normal*100:.1f}%")
print()
print("Matriz de confusión (filas = real, columnas = predicho):")
print(confusion_matrix(y_test, pred_normal))

### 🤔 ¿Qué acaba de pasar?

- Aunque Gradient Boosting es un modelo mucho más sofisticado que el "modelo perezoso", por defecto **también tiende a favorecer a la clase mayoritaria**: como el 90% de los ejemplos de entrenamiento son "sanos", minimizar el error total empuja al modelo a acertar sobre todo en esa clase, incluso si eso significa fallarle a varios enfermos.
- La matriz de confusión te deja ver esto en detalle: la fila de "enfermos reales" (clase 1) probablemente tiene bastantes casos que el modelo predijo como "sano" (falsos negativos) — justo los casos que más nos importaría detectar en un examen médico real.
- El recall de esta versión normal seguramente ya es mejor que el 0% del modelo perezoso (porque sí aprendió algo de los indicadores), pero puede seguir siendo más bajo de lo que nos gustaría para un problema donde no detectar un caso real es grave.

---
## 6. La solución: `class_weight='balanced'` ⚖️

La idea es simple: si hay 9 pacientes sanos por cada enfermo, le decimos al modelo *"cada vez que te equivoques con un paciente enfermo, ese error va a pesar 9 veces más que equivocarte con uno sano"*. Así el modelo ya no puede "hacer trampa" ignorando a la clase minoritaria — le sale caro hacerlo.

En Scikit-Learn, esto se activa con el parámetro `class_weight='balanced'`, disponible en modelos basados en árboles como `DecisionTreeClassifier` o `RandomForestClassifier` (nota: `GradientBoostingClassifier` no tiene este parámetro directamente, así que aquí lo mostramos con un `RandomForestClassifier` — el mismo bosque del cuaderno anterior — para comparar "antes y después" de forma clara).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Sin balancear (igual que antes: cada paciente pesa lo mismo)
bosque_normal = RandomForestClassifier(n_estimators=200, random_state=42)
bosque_normal.fit(X_train, y_train)
recall_bosque_normal = recall_score(y_test, bosque_normal.predict(X_test))

# Balanceado: los errores en la clase minoritaria pesan más
bosque_balanceado = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
bosque_balanceado.fit(X_train, y_train)
recall_bosque_balanceado = recall_score(y_test, bosque_balanceado.predict(X_test))

print(f"Random Forest normal      -> Recall (enfermos detectados): {recall_bosque_normal*100:.1f}%")
print(f"Random Forest balanceado  -> Recall (enfermos detectados): {recall_bosque_balanceado*100:.1f}%")

### 🤔 ¿Qué acaba de pasar?

- Entrenamos dos bosques idénticos, con la única diferencia de que uno usa `class_weight='balanced'`.
- Ese parámetro le dice al modelo que, al medir qué tan "pura" queda cada división del árbol, cuente los errores sobre pacientes enfermos como si hubiera muchos más de ellos (compensando el 9-a-1 real). El modelo entonces se esfuerza más por encontrar patrones que sí distingan a los enfermos, en vez de simplemente ignorarlos.
- Compara los dos valores de recall: normalmente el bosque balanceado detecta una proporción notablemente mayor de los pacientes enfermos. Es posible que a cambio la exactitud general baje un poco — y está bien, porque ya sabemos que la exactitud sola era una métrica engañosa aquí. Lo que ganamos es justo lo que más nos importaba: detectar más casos reales de la condición.
- Esta idea se aplica igual en detección de fraude, predicción de fallas de máquinas, diagnóstico médico, o cualquier problema donde el caso "interesante" es raro pero costoso de no detectar.

---
## 7. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Boosting | Entrena árboles uno tras otro (relevo secuencial), cada uno corrigiendo los errores del anterior. |
| Árbol débil (*stump*) | Un árbol muy simple (a menudo `max_depth=1` a `3`) usado como bloque básico dentro de Boosting. |
| `GradientBoostingClassifier` | Clase de Scikit-Learn que implementa Boosting sumando las correcciones de muchos árboles débiles. |
| Dataset desbalanceado | Un problema donde una clase (ej. 90%) es mucho más común que la otra (ej. 10%). |
| Trampa de la exactitud | Con clases desbalanceadas, un modelo puede tener exactitud alta y aun así no detectar casi ningún caso de la clase rara. |
| Recall de la clase minoritaria | Mide qué proporción de los casos "raros" reales fue detectada — suele ser más honesto que la exactitud en estos problemas. |
| `class_weight='balanced'` | Hace que los errores sobre la clase minoritaria pesen más durante el entrenamiento, mejorando su detección. |

➡️ **Siguiente paso:** con esto cierras el módulo 09 de Árboles de Decisión y sus ensambles. En el próximo módulo del curso continuarás con el módulo de **Clustering**, donde en vez de predecir una etiqueta que ya conocemos, aprenderás a descubrir **grupos ocultos** dentro de los datos sin que nadie te diga de antemano cuáles son.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>